# 🌿 Grass Plant Auto-Annotator
**Model:** Grounding DINO (IDEA-Research/grounding-dino-base) from HuggingFace  
**Output:** Annotated images + Bounding boxes in YOLO & COCO format  
**IDE:** Kaggle (GPU T4 x2 recommended)


## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate timm supervision Pillow opencv-python-headless matplotlib

## 2. Imports & Setup

In [ ]:
import os
import json
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import supervision as sv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Load Grounding DINO Model (Best Zero-Shot Detector)

In [ ]:
# Using Grounding DINO Base — best balance of speed and accuracy
# Switch to 'IDEA-Research/grounding-dino-tiny' if running on CPU/low VRAM

MODEL_ID = "IDEA-Research/grounding-dino-base"

print(f"Loading model: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print("✅ Model loaded successfully!")

## 4. Configure Detection Prompts & Thresholds

In [ ]:
# ── TUNABLE SETTINGS ─────────────────────────────────────────────────────────

# Text prompt — describe your grass plants. Be specific for better accuracy.
# Grounding DINO uses natural language, so try variations like:
#   "grass plant", "weed", "paddy plant", "rice plant", "crop plant"
TEXT_PROMPT = "grass plant . weed plant . grass"

# Confidence thresholds (lower = more detections, higher = more precise)
BOX_THRESHOLD = 0.30     # Minimum confidence for bounding boxes
TEXT_THRESHOLD = 0.25    # Minimum confidence for text matching

# Output format: "yolo", "coco", or "both"
OUTPUT_FORMAT = "both"

# Class ID for grass plants (used in YOLO format)
CLASS_ID = 0
CLASS_NAME = "grass_plant"

# Output directory
OUTPUT_DIR = Path("/kaggle/working/annotations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "annotated_images").mkdir(exist_ok=True)
(OUTPUT_DIR / "labels_yolo").mkdir(exist_ok=True)

print(f"✅ Config ready. Output → {OUTPUT_DIR}")

## 5. Core Detection & Annotation Functions

In [ ]:
def detect_grass_plants(image: Image.Image, text_prompt: str) -> dict:
    """
    Run Grounding DINO on a PIL image.
    Returns raw model outputs.
    """
    inputs = processor(
        images=image,
        text=text_prompt,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    # Post-process → absolute pixel coordinates
    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs["input_ids"],
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]]  # (H, W)
    )[0]

    return results


def results_to_bboxes(results: dict, img_w: int, img_h: int) -> list:
    """
    Convert model results → list of bbox dicts.
    Each dict: {
        'xyxy': [x1, y1, x2, y2],         # absolute pixels
        'xywh': [x, y, w, h],             # absolute pixels
        'yolo': [cx, cy, w, h],           # normalized 0-1
        'confidence': float,
        'label': str
    }
    """
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results["labels"]

    bboxes = []
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        w = x2 - x1
        h = y2 - y1
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        bboxes.append({
            "xyxy": [float(x1), float(y1), float(x2), float(y2)],
            "xywh": [float(x1), float(y1), float(w), float(h)],
            "yolo": [
                round(cx / img_w, 6),
                round(cy / img_h, 6),
                round(w / img_w, 6),
                round(h / img_h, 6)
            ],
            "confidence": float(score),
            "label": label
        })

    return bboxes


def draw_annotations(image: Image.Image, bboxes: list) -> Image.Image:
    """
    Draw bounding boxes + confidence scores on image.
    Returns annotated PIL Image.
    """
    annotated = image.copy().convert("RGB")
    draw = ImageDraw.Draw(annotated)

    colors = [
        "#00FF41", "#FF6B35", "#FFD700", "#00D4FF",
        "#FF1493", "#7FFF00", "#FF4500", "#1E90FF"
    ]

    for i, bbox in enumerate(bboxes):
        x1, y1, x2, y2 = bbox["xyxy"]
        color = colors[i % len(colors)]
        conf = bbox["confidence"]

        # Draw box (thick border)
        for thickness in range(3):
            draw.rectangle(
                [x1 - thickness, y1 - thickness, x2 + thickness, y2 + thickness],
                outline=color
            )

        # Label background
        label_text = f"{CLASS_NAME} {conf:.2f}"
        text_bbox = draw.textbbox((x1, y1 - 20), label_text)
        draw.rectangle(text_bbox, fill=color)
        draw.text((x1, y1 - 20), label_text, fill="black")

    return annotated


print("✅ Functions defined.")

## 6. Export Functions (YOLO & COCO)

In [ ]:
def save_yolo_labels(bboxes: list, save_path: Path):
    """
    Save YOLO format: <class_id> <cx> <cy> <w> <h>  (all normalized)
    """
    with open(save_path, "w") as f:
        for bbox in bboxes:
            cx, cy, w, h = bbox["yolo"]
            f.write(f"{CLASS_ID} {cx} {cy} {w} {h}\n")
    print(f"  YOLO → {save_path}")


def build_coco_annotation(image_id: int, bboxes: list, img_w: int, img_h: int,
                           filename: str, ann_id_start: int = 0) -> tuple:
    """
    Build COCO-format image + annotation entries.
    """
    image_entry = {
        "id": image_id,
        "file_name": filename,
        "width": img_w,
        "height": img_h
    }
    ann_entries = []
    for i, bbox in enumerate(bboxes):
        x, y, w, h = bbox["xywh"]
        ann_entries.append({
            "id": ann_id_start + i,
            "image_id": image_id,
            "category_id": CLASS_ID,
            "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
            "area": round(w * h, 2),
            "iscrowd": 0,
            "score": round(bbox["confidence"], 4)
        })
    return image_entry, ann_entries


print("✅ Export functions ready.")

## 7. Main Pipeline — Process All Images

In [ ]:
# ── INPUT IMAGES ─────────────────────────────────────────────────────────────
# Option A: Kaggle Dataset → set your dataset path
IMAGE_DIR = Path("/kaggle/input")   # ← change to your dataset folder

# Option B: Working directory uploads
# IMAGE_DIR = Path("/kaggle/working")

# Supported image extensions
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"]

image_paths = []
for ext in IMAGE_EXTS:
    image_paths.extend(IMAGE_DIR.rglob(f"*{ext}"))
    image_paths.extend(IMAGE_DIR.rglob(f"*{ext.upper()}"))

print(f"Found {len(image_paths)} image(s) to annotate:")
for p in image_paths:
    print(f"  {p}")

In [ ]:
# ── PROCESS IMAGES ────────────────────────────────────────────────────────────

all_results = {}   # image_name → list of bboxes
coco_images = []
coco_annotations = []
ann_id_counter = 0

for img_id, img_path in enumerate(image_paths):
    print(f"\n[{img_id+1}/{len(image_paths)}] Processing: {img_path.name}")

    # Load image
    image = Image.open(img_path).convert("RGB")
    img_w, img_h = image.size
    print(f"  Size: {img_w}×{img_h}")

    # Run Grounding DINO
    results = detect_grass_plants(image, TEXT_PROMPT)
    bboxes = results_to_bboxes(results, img_w, img_h)
    print(f"  Detected: {len(bboxes)} grass plant(s)")

    if len(bboxes) == 0:
        print("  ⚠️  No detections. Try lowering BOX_THRESHOLD or changing TEXT_PROMPT.")

    all_results[img_path.name] = bboxes

    # Draw & save annotated image
    annotated = draw_annotations(image, bboxes)
    out_img_path = OUTPUT_DIR / "annotated_images" / f"annotated_{img_path.name}"
    annotated.save(out_img_path)
    print(f"  Annotated image → {out_img_path}")

    # Save YOLO labels
    if OUTPUT_FORMAT in ("yolo", "both"):
        yolo_path = OUTPUT_DIR / "labels_yolo" / f"{img_path.stem}.txt"
        save_yolo_labels(bboxes, yolo_path)

    # Build COCO entries
    if OUTPUT_FORMAT in ("coco", "both"):
        img_entry, ann_entries = build_coco_annotation(
            image_id=img_id,
            bboxes=bboxes,
            img_w=img_w,
            img_h=img_h,
            filename=img_path.name,
            ann_id_start=ann_id_counter
        )
        coco_images.append(img_entry)
        coco_annotations.extend(ann_entries)
        ann_id_counter += len(ann_entries)

print("\n✅ All images processed!")

## 8. Save COCO JSON

In [ ]:
if OUTPUT_FORMAT in ("coco", "both"):
    coco_dict = {
        "info": {
            "description": "Grass Plant Auto-Annotation via Grounding DINO",
            "version": "1.0",
            "model": "IDEA-Research/grounding-dino-base",
            "prompt": TEXT_PROMPT,
            "box_threshold": BOX_THRESHOLD,
            "text_threshold": TEXT_THRESHOLD
        },
        "categories": [
            {"id": CLASS_ID, "name": CLASS_NAME, "supercategory": "plant"}
        ],
        "images": coco_images,
        "annotations": coco_annotations
    }

    coco_path = OUTPUT_DIR / "annotations_coco.json"
    with open(coco_path, "w") as f:
        json.dump(coco_dict, f, indent=2)

    print(f"COCO JSON → {coco_path}")
    print(f"  Total images: {len(coco_images)}")
    print(f"  Total annotations: {len(coco_annotations)}")

## 9. Print All Bounding Box Vectors

In [ ]:
print("=" * 70)
print("BOUNDING BOX VECTORS SUMMARY")
print("=" * 70)

for img_name, bboxes in all_results.items():
    print(f"\n📷 {img_name}  ({len(bboxes)} detections)")
    print(f"  {'#':<4} {'[x1, y1, x2, y2] (pixels)':<35} {'YOLO [cx,cy,w,h] (norm)':<35} {'Conf':<6}")
    print(f"  {'-'*80}")
    for i, bbox in enumerate(bboxes):
        xyxy = [f"{v:.1f}" for v in bbox['xyxy']]
        yolo = [f"{v:.4f}" for v in bbox['yolo']]
        print(f"  {i:<4} {str(xyxy):<35} {str(yolo):<35} {bbox['confidence']:.3f}")

print("\n" + "=" * 70)

## 10. Visualize Results

In [ ]:
annotated_images = list((OUTPUT_DIR / "annotated_images").glob("*.jpg")) + \
                   list((OUTPUT_DIR / "annotated_images").glob("*.png"))

n = min(len(annotated_images), 6)  # show up to 6
if n == 0:
    print("No annotated images to display.")
else:
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
    if n == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]

    for idx, img_path in enumerate(annotated_images[:n]):
        r, c = divmod(idx, cols)
        img = Image.open(img_path)
        axes[r][c].imshow(img)
        axes[r][c].set_title(img_path.name, fontsize=10)
        axes[r][c].axis("off")

    # Hide unused subplots
    for idx in range(n, rows * cols):
        r, c = divmod(idx, cols)
        axes[r][c].axis("off")

    plt.suptitle("Grass Plant Annotations — Grounding DINO", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "preview.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Preview saved!")

## 11. Generate YOLO dataset.yaml

In [ ]:
yaml_content = f"""# YOLO Dataset Config — Grass Plant Detection
path: /kaggle/working/annotations
train: annotated_images
val: annotated_images  # replace with actual val split

nc: 1  # number of classes
names:
  0: {CLASS_NAME}

# Annotated using Grounding DINO (IDEA-Research/grounding-dino-base)
# Prompt: {TEXT_PROMPT}
# Box Threshold: {BOX_THRESHOLD}
"""

yaml_path = OUTPUT_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"YOLO dataset.yaml → {yaml_path}")
print(yaml_content)

## 12. Summary

In [ ]:
total_detections = sum(len(v) for v in all_results.values())

print("=" * 50)
print("🌿 ANNOTATION COMPLETE")
print("=" * 50)
print(f"Images processed  : {len(all_results)}")
print(f"Total detections  : {total_detections}")
print(f"Output directory  : {OUTPUT_DIR}")
print()
print("Files generated:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  {str(f.relative_to(OUTPUT_DIR)):<50} {size:.1f} KB")
print("=" * 50)